In [1]:
import torch
from torch.utils.data import Dataset
import pandas as pd
import numpy as np

class StockDataset(Dataset):
    """
    工业级股票数据集类
    目标：将量价数据转化为深度学习张量
    """
    def __init__(self, df, window_size=30):
        """
        __init__: 初始化数据，就像是给 AI 准备教材
        :param df: 包含量价数据的 Pandas DataFrame
        :param window_size: 训练窗口长度，比如用过去30天的数据预测明天
        """
        # 1. 金融数据处理核心：确保数据是按日期排序的
        self.df = df.sort_values('date').reset_index(drop=True)
        self.window_size = window_size
        
        # 2. 特征工程：只提取模型需要的数值列（如：开高低收、技术指标）
        # 假设我们只用收盘价和成交量，实际开发中这里会包含上百个特征
        self.features = self.df[['close', 'volume']].values
        
        # 3. 目标值：预测下一天的收益率 (yReg)
        # 这里的计算逻辑体现了金融逻辑：(明天价格 - 今天价格) / 今天价格
        self.target = self.df['close'].pct_change().shift(-1).values

    def __len__(self):
        """
        __len__: 告诉 PyTorch 这一共有多少个“学习案例”
        """
        return len(self.df) - self.window_size - 1

    def __getitem__(self, idx):
        """
        __getitem__: 最核心的方法。模拟面试官必考：如何根据索引抓取一段连续的时间序列数据
        """
        # 抓取从 idx 到 idx + window_size 的特征矩阵
        x = self.features[idx : idx + self.window_size]
        
        # 抓取对应的预测目标（第 idx + window_size 天的收益率）
        y = self.target[idx + self.window_size]
        
        # 关键动作：转化为 PyTorch 张量 (Tensor)
        return torch.FloatTensor(x), torch.FloatTensor([y])

# --- 模拟运行 ---
# 如果你手头有数据，可以直接传入测试：
# train_ds = StockDataset(your_dataframe)
# print(f"第一个案例的形状: {train_ds[0][0].shape}")

In [ ]:
!pip install torch

In [ ]:
!pip install torch -i https://pypi.tuna.tsinghua.edu.cn/simple --default-timeout=1000

In [2]:
import torch
print(torch.__version__)

2.8.0+cpu


In [ ]:
!pip install --upgrade numexpr bottleneck -i https://pypi.tuna.tsinghua.edu.cn/simple

In [ ]:
import pandas as pd
import numpy as np

# 1. 凭空捏造一份假数据（假装我们抓取了某只股票 50 天的行情）
print("--- 1. 正在生成测试用的假数据 ---")
dummy_data = {
    'date': pd.date_range(start='2020-01-01', periods=50),
    'close': np.random.rand(50) * 100,            # 随机生成50天的收盘价
    'volume': np.random.randint(1000, 5000, 50)   # 随机生成50天的成交量;两个特征数
}
my_dataframe = pd.DataFrame(dummy_data)
print("假数据造好了！前3天长这样：\n", my_dataframe.head(3), "\n")


# 2. 拿出你刚才存在电脑里的图纸，把假数据喂进去！（实例化对象）
print("--- 2. 把数据喂给你写的 StockDataset 图纸 ---")
train_ds = StockDataset(my_dataframe, window_size=30)#time step
print(f"按照每次看30天的规则，这50天的数据一共能切出 {len(train_ds)} 个学习案例。\n")


# 3. 抽查第一个案例，看看你的 __getitem__ 魔术方法灵不灵！
print("--- 3. 抽查第 1 个切片给面试官看 ---")
x, y = train_ds[0]  # 这就是在调用你写的 __getitem__(0)

print(f"喂给 AI 的特征矩阵 (X) 的形状是: {x.shape}") 
print(f" -> 解释：这就代表提取了 30 天的数据，每天有 2 个特征（收盘价和成交量）。")
print(f"\n需要 AI 预测的目标收益率 (Y) 是: {y.item():.4f}")

#结果中的[30,2]是PyTorch 张量（Tensor）维度。30（Sequence Length / 时间步）：代表 AI 一次回头看 30 天的历史走势。
#2（Feature Dimension / 特征数）： 代表你每天喂给 AI 几个维度的信息。

In [ ]:
!git config --global --unset http.proxy
!git config --global --unset https.proxy

In [ ]:
# 1. 关掉安全证书验证（最常见的连不上原因）
!git config --global http.sslVerify false

# 2. 把超时时间拉长到 10 分钟（600秒）
!git config --global http.lowSpeedLimit 0
!git config --global http.lowSpeedTime 999999

# 3. 增大传输缓存到 500MB
!git config --global http.postBuffer 524288000

In [ ]:
!git push -u origin main

In [ ]:
import torch
from torch.utils.data import DataLoader

# 设定每次拿 16 条数据，并且打乱顺序
train_loader = DataLoader(dataset=train_ds, batch_size=16, shuffle=True)

# (这里用 iter 和 next 模拟一次提取)
batch_x, batch_y = next(iter(train_loader))

print(f"这一批数据打包好了")
print(f"输入特征 (X) 的形状是: {batch_x.shape}") 
print(f"目标预测 (Y) 的形状是: {batch_y.shape}")

# 形状是 [16, 30, 2]：
# 16 代表 batch_size (一次拿了16个样本)
# 30 代表 window_size (每个样本包含过去30天的数据)
# 2 代表 特征数 (收盘价和成交量)
# 1 代表 AI 需要对每个案例输出 1 个预测结果